# DHWS-MRI-Best — K-Space MRI Reconstruction on Google Colab

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells in order

**Data (automatic — no registration needed):**
- Cell 4 installs `datasets` and auto-downloads fastMRI knee subset from HuggingFace
- Falls back to synthetic Shepp-Logan phantoms if download fails

**Outputs** saved to `MyDrive/DHWS/outputs_mri_best/`

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/DHWS/outputs_mri_best', exist_ok=True)
os.makedirs('/content/drive/MyDrive/DHWS/data/mri',        exist_ok=True)
print('Drive mounted.')
print('Outputs → /content/drive/MyDrive/DHWS/outputs_mri_best/')

In [ ]:
# ── Cell 2: Check GPU ────────────────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# datasets  — HuggingFace auto-download of fastMRI (no registration needed)
# h5py      — read local fastMRI .h5 files if you have them
!pip install -q datasets h5py
print('Dependencies installed.')

In [ ]:
# ── Cell 4: Upload dhws_mri_best.py ──────────────────────────────────────────
# Option A — upload from your computer
from google.colab import files
uploaded = files.upload()   # select dhws_mri_best.py from Desktop/DHWS

# Option B — copy from Drive if already uploaded:
# import shutil
# shutil.copy('/content/drive/MyDrive/DHWS/dhws_mri_best.py', '/content/dhws_mri_best.py')

In [ ]:
# ── Cell 5: Patch config for Colab + GPU ─────────────────────────────────────
with open('dhws_mri_best.py', 'r') as f:
    src = f.read()

# Redirect outputs to Google Drive
src = src.replace(
    "data_root:      Path  = Path('./data/mri')",
    "data_root:      Path  = Path('/content/drive/MyDrive/DHWS/data/mri')"
)
src = src.replace(
    "out_dir:        Path  = Path('./outputs_mri_best')",
    "out_dir:        Path  = Path('/content/drive/MyDrive/DHWS/outputs_mri_best')"
)

# GPU-optimised batch size (2 CPU → 8 GPU)
# 320x320 grayscale MRI — 8 fits comfortably in 15 GB T4 VRAM
src = src.replace(
    'batch_size:     int   = 2',
    'batch_size:     int   = 8'
)

# More epochs on GPU
src = src.replace(
    'epochs:         int   = 50',
    'epochs:         int   = 100'
)

with open('colab_mri_model.py', 'w') as f:
    f.write(src)

print('Config patched:')
print('  data_root  → /content/drive/MyDrive/DHWS/data/mri')
print('  out_dir    → /content/drive/MyDrive/DHWS/outputs_mri_best')
print('  batch_size → 8')
print('  epochs     → 100')

In [ ]:
# ── Cell 6: Train ────────────────────────────────────────────────────────────
# On T4 GPU with synthetic data: ~30-60 s/epoch  (100 epochs ≈ 1-2 hours)
# On T4 GPU with fastMRI HF:     ~2-5 min/epoch  (100 epochs ≈ 4-8 hours)
#
# Startup message will tell you which data source was found.
# Zero-filled baseline is printed before training starts —
# the model should exceed it by epoch 1-2.

%cd /content
%run colab_mri_model.py

In [ ]:
# ── Cell 7: Visualise reconstructions ────────────────────────────────────────
# Each row in the saved PNG: [zero_filled | spectral | fused | refined | target]

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import glob

out_dir  = Path('/content/drive/MyDrive/DHWS/outputs_mri_best')
samples  = sorted(glob.glob(str(out_dir / 'samples_ep*.png')))

if not samples:
    print('No sample images yet — training may still be running.')
else:
    show = samples[-3:] if len(samples) >= 3 else samples   # last 3 checkpoints
    fig, axes = plt.subplots(1, len(show), figsize=(6 * len(show), 6))
    if len(show) == 1: axes = [axes]
    for ax, p in zip(axes, show):
        ax.imshow(mpimg.imread(p), cmap='gray')
        ax.set_title(Path(p).stem, fontsize=10)
        ax.axis('off')
    plt.suptitle('Rows: zero-filled | spectral | fused | refined | ground truth', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 8: Plot training curves ─────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

log_path = '/content/drive/MyDrive/DHWS/outputs_mri_best/training_log.csv'
try:
    log = pd.read_csv(log_path)
except FileNotFoundError:
    print('training_log.csv not found — training still running or not finished.')
    raise

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(log['epoch'], log['train_loss'], label='Train')
axes[0].plot(log['epoch'], log['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(log['epoch'], log['psnr_db'], color='green')
axes[1].set_title('PSNR (dB)'); axes[1].grid(True)

axes[2].plot(log['epoch'], log['ssim'], color='blue')
axes[2].set_title('SSIM'); axes[2].grid(True)

axes[3].plot(log['epoch'], log['nmse'], color='red')
axes[3].set_title('NMSE'); axes[3].grid(True)

plt.tight_layout()
plt.show()

best = log.loc[log['psnr_db'].idxmax()]
print(f"Best epoch : {int(best['epoch'])}")
print(f"Best PSNR  : {best['psnr_db']:.2f} dB")
print(f"Best SSIM  : {best['ssim']:.4f}")
print(f"Best NMSE  : {best['nmse']:.5f}")

In [ ]:
# ── Cell 9: Download best checkpoint ─────────────────────────────────────────
from google.colab import files
files.download('/content/drive/MyDrive/DHWS/outputs_mri_best/best_model.pt')